---

# Tugas Besar 2 IF3170 Inteligensi Artifisial

*This notebook is the assignment template. Each group works on its own copy, and cells or sections may be added as needed.*

---

## Group XX

- Name (NIM)
- Name (NIM)
- Name (NIM)
- Name (NIM)
- Name (NIM)

---

## Table of Contents

1. [**Introduction**](#1)
2. [**Initialization**](#2)
3. [**Data Understanding (Optional)**](#3)
4. [**Training and Validation Split**](#4)
5. [**Data Preparation**](#5)
6. [**Preparation Pipeline**](#6)
7. [**Modeling and Validation**](#7)
8. [**Comparison with scikit-learn**](#8)
9. [**Saving and Loading Models**](#9)
10. [**Kaggle Submission**](#10)
11. [**Error Analysis**](#11)


---

# Introduction <a name="1"></a>

---

This section frames the problem in the words of the group.

## Overview

*To be written by the group.*

## Aim

*To be written by the group.*

## Metric

*To be written by the group.*

## Dataset

*To be written by the group.*

## Approach

*To be written by the group.*

---

# Initialization <a name="2"></a>

---

This section prepares the environment, the settings, and the data files.

## Environment Setup

The following cell lists the base libraries. The install line is commented by default.

In [ ]:
# %pip install -q numpy pandas

## Import Libraries

Every import of the notebook is kept in this cell.

In [ ]:
import os
import random
from pathlib import Path

import sys

import numpy as np
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))

from eda import compare_train_test, describe_data, explore_data, verify_quality
from evaluation import compare_with_sklearn, cross_validate, scores
from naive_bayes import NaiveBayes
from nb_compare import SklearnCategoricalNB, exact_equivalence
from preprocessing import PreparationPipeline, construct_features, drop_duplicates_and_invalid, split_train_val

## Settings

Paths and shared constants are kept in this class.

In [ ]:
class Settings:
    SEED       = 42
    DATA_DIR   = Path('data')
    OUTPUT_DIR = Path('output')

CFG = Settings()
CFG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Seed Everything

A fixed seed keeps the results reproducible.

In [ ]:
def seed_everything(seed: int):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(CFG.SEED)

## Load Dataset

The competition files are read here. The `id` column stays as text because some values resemble numbers.

In [ ]:
train_df = pd.read_csv(CFG.DATA_DIR / 'train.csv', dtype={'id': str})
test_df  = pd.read_csv(CFG.DATA_DIR / 'test.csv', dtype={'id': str})

print(f'Train: {train_df.shape}  Test: {test_df.shape}')

---

# Data Understanding (Optional) <a name="3"></a>

---

Data understanding builds the knowledge that later preparation decisions rest on.

## Describe Data

This part covers the size, the attribute types, and the meaning of each column.

In [ ]:
eda_describe = describe_data(train_df)

### Insights

- `train.csv` has 27,373 rows with 17 attributes plus the target `is_canceled` (the `id` column is only a label). `test.csv` has 6,844 rows without the target.
- Eight attributes have missing values, from 2.96% (`num_adults`) to 11.09% (`num_children`). In total 10,737 rows (39.2%) miss at least one value, so dropping incomplete rows would discard a large part of the data.
- 32.77% of the bookings are canceled. A model that always predicts "not canceled" reaches an accuracy of 0.6723, which is the baseline to beat.
- Some categories are very rare: `meal_package` has only 4 rows of Full Board and `room_category` has 7 rows of Type E. Smoothing is needed so that such a category never gets a probability of zero.
- Arrivals cover two years only (`arrival_year` is 2023 or 2024).

## Explore Data

This part covers distributions and relationships between attributes.

In [ ]:
eda_target_corr = explore_data(train_df, CFG.OUTPUT_DIR / 'eda')
eda_target_corr.head(10)

### Insights

- `days_before_arrival` has the strongest linear relation with cancellation (+0.440): the longer the lead time, the more likely the booking is canceled.
- `special_requests` follows with -0.258: guests who make requests cancel less. Next are `arrival_year` (+0.180), `nightly_rate_idr` (+0.134) and `is_returning_guest` (-0.107).
- Correlation only measures linear relations between numeric columns. The three categorical attributes are read from the cancel-rate charts above instead.

### Train Versus Test

The test set is used here for diagnosis only. Nothing is fitted from it.

In [ ]:
eda_shift = compare_train_test(train_df, test_df)

### Insights

- No sign of distribution shift between train and test. The largest Kolmogorov-Smirnov distance of a numeric attribute is 0.0149 (`arrival_day`), and the total variation distance of every categorical attribute is at most 0.0048.
- Every category in the test set also appears in the training set, and the share of missing values differs by less than 0.5 percentage point in each column.
- This check looks at one attribute at a time and cannot see a change in how attributes combine. It supports, but does not prove, that validation scores from `train.csv` carry over to the test set.

## Verify Data Quality

This part covers completeness, consistency, and the validity of values.

In [ ]:
eda_quality_train = verify_quality(train_df, 'train')
eda_quality_test = verify_quality(test_df, 'test')

### Insights

- Train: 10,737 rows with at least one missing value, 66 extra copies of duplicated rows (ignoring `id`), of which 2 groups carry conflicting labels, and 64 rows with zero nights. No row has zero guests. Test: 2,669 rows with missing values, 3 extra duplicate copies and 9 zero-night rows.
- The IQR rule flags many outliers in skewed counts (for example 2,020 in `num_children`, 916 in `days_before_arrival`, 1,144 in `nightly_rate_idr`). For attributes that are mostly zero this rule is weak, so the flags are a hint to inspect and not a reason to delete rows.
- `drop_duplicates_and_invalid` in `src/preprocessing.py` removes conflicting duplicate groups and extra copies and keeps zero-night rows. The Naive Bayes model cuts numeric attributes into quantile bins, so extreme values only fall in the outer bins and do not distort the estimates.

---

# Training and Validation Split <a name="4"></a>

---

The split comes before data preparation so that validation data stays unseen. Only `train.csv` is split, and `test.csv` serves the Kaggle submission alone.

In [ ]:
# train_set, val_set = ...

---

# Data Preparation <a name="5"></a>

---

Each process below applies where the data calls for it, and the Insights block under it records the reason for the decision.

Preparation steps may use `sklearn`. Every statistic comes from the training split only, then transforms the other sets.

## Select Data

Selection decides which records and attributes stay in the analysis.

In [ ]:
# Write your code here

## Clean Data

Cleaning raises data quality to the level the chosen models need.

### Missing Values

This part covers missing values, if any.

In [ ]:
# Write your code here

### Duplicates

This part covers repeated records.

In [ ]:
# Write your code here

### Outliers

This part covers values far from the rest of the data.

In [ ]:
# Write your code here

### Data Errors

This part covers wrong types, inconsistent labels, and impossible values.

In [ ]:
# Write your code here

## Construct Data

Construction derives attributes and transforms values for modeling.

### Attribute Construction

<!-- rubric v3: derived columns, rate_per_guest_night -->

This part covers new attributes derived from existing ones.

In [ ]:
# Write your code here

### Feature Scaling

This part covers normalization to the range 0 to 1 and standardization.

In [ ]:
# Write your code here

### Encoding

This part covers the numeric representation of categorical attributes.

In [ ]:
# Write your code here

### Dimensionality Reduction

This part covers fewer dimensions that keep the meaning of the data.

In [ ]:
# Write your code here

## Class Balance

This part covers the proportion of each class in the training data.

In [ ]:
# Write your code here

## Format Data

Formatting produces the structure the models read, such as a numeric matrix.

In [ ]:
# Write your code here

---

# Preparation Pipeline <a name="6"></a>

---

<!-- rubric v3: fitted steps, fitted_prep_steps -->

The pipeline combines the preparation steps so they run in one consistent order.

In [ ]:
# Write your code here

---

# Modeling and Validation <a name="7"></a>

---

The target is `is_canceled`. The models here are implemented from scratch, with math libraries such as NumPy.

## K-Nearest Neighbors

<!-- rubric v3: pairwise distances, val_neighbor_dists -->

This part covers the nearest neighbor classifier and its validation.

In [ ]:
# Write your code here

## Decision Tree Learning

This part covers one chosen variant, which is ID3, C4.5, or CART, and its validation.

In [ ]:
# Write your code here

## Logistic Regression

This part covers the model with L1, L2, and Elastic Net regularization.

In [ ]:
# Write your code here

## Support Vector Machine

<!-- rubric v3: total slack, slack_total -->

This part covers the linear, polynomial, and Gaussian kernels.

In [ ]:
# Write your code here

## Naive Bayes

This part covers the Naive Bayes classifier and its validation.

In [ ]:
nb_train, nb_val = split_train_val(train_df, seed=CFG.SEED)

def prepare_nb(train_part, val_part, steps=()):
    train_part, _ = drop_duplicates_and_invalid(train_part)
    pipeline = PreparationPipeline(list(steps)).fit(train_part)
    return (*pipeline.transform(train_part), *pipeline.transform(val_part))

nb_X_tr, nb_y_tr, nb_X_va, nb_y_va = prepare_nb(nb_train, nb_val)
nb_model = NaiveBayes(alpha=1.0, n_bins=20).fit(nb_X_tr, nb_y_tr)
nb_val_proba = nb_model.predict_proba(nb_X_va)

print(f'Train: {nb_X_tr.shape}  Validation: {nb_X_va.shape}')
print(f'Majority-class accuracy on validation: {1 - nb_y_va.mean():.4f}')
pd.Series(scores(nb_y_va, nb_val_proba), name='validation')

### Insights

- The stratified 80/20 split (seed 42) gives 21,898 training rows. After removing duplicates 21,856 remain, and 5,475 rows are held out for validation.
- On validation the model reaches an accuracy of 0.7784 against the baseline of 0.6723, with an F1 of 0.6452 and a ROC-AUC of 0.8364.
- Precision (0.6788) is higher than recall (0.6148), so at the threshold of 0.5 the model misses part of the real cancellations. ROC-AUC is higher than what the thresholded scores suggest, which means the ranking of the probabilities is good and the threshold can be moved if the cost of errors is not equal.

### Cross-Validation and Feature Choice

The same stratified 5-fold split (seed 42) is used for every model, and preparation is refitted inside each fold. The comparison asks whether the derived columns of `construct_features` help Naive Bayes.

In [ ]:
nb_cv_raw = cross_validate(lambda: NaiveBayes(n_bins=20), prepare_nb, train_df)
nb_cv_derived = cross_validate(
    lambda: NaiveBayes(n_bins=20),
    lambda a, b: prepare_nb(a, b, steps=(construct_features,)),
    train_df,
)
pd.concat({'raw features': nb_cv_raw.loc['mean'], 'with derived features': nb_cv_derived.loc['mean']}, axis=1).T

### Insights

- With the 17 original attributes the 5-fold mean is an accuracy of 0.7812, an F1 of 0.6462 and a ROC-AUC of 0.8347.
- Adding the columns of `construct_features` lowers accuracy to 0.7677 (-0.0134), precision to 0.6489 and ROC-AUC to 0.8224, and only recall rises (0.6342 against 0.6099).
- A likely reason is that the derived columns are computed from attributes that are already in the model, and Naive Bayes assumes the attributes are independent given the class, so the same evidence is counted twice. This reason was not tested separately.
- Decision: Naive Bayes uses the original attributes, with an empty list of preparation steps.

### Number of Bins

Numeric attributes with more than 10 distinct values are cut into quantile bins. The number of bins is chosen from training folds only.

In [ ]:
nb_bins_grid = [5, 10, 20, 30, 40]
nb_bins_cv = pd.DataFrame({
    b: cross_validate(lambda b=b: NaiveBayes(n_bins=b), prepare_nb, train_df).loc['mean']
    for b in nb_bins_grid
}).T
nb_bins_cv.index.name = 'n_bins'
nb_bins_cv

### Insights

- Going from 5 to 20 bins raises the ROC-AUC from 0.8227 to 0.8347, and accuracy from 0.7814 to 0.7812 stays flat, so the gain is in the ranking of probabilities.
- Beyond 20 bins the gain is about 0.001 or less (ROC-AUC 0.8355 and accuracy 0.7821 at 40 bins). The model keeps 20 bins because the extra bins buy almost nothing.

## Model Evaluation

<!-- figure palette: #6d2fa3 -->

This part compares the validation results of all models.

In [ ]:
# Write your code here

## Improvements (Optional)

This part covers hyperparameter tuning, cross-validation, and further visualization.

In [ ]:
# Write your code here

---

# Comparison with scikit-learn <a name="8"></a>

---

This section compares each from-scratch model with its `scikit-learn` counterpart.

In [ ]:
# Write your code here

### Naive Bayes

The scikit-learn counterpart is `CategoricalNB` with `KBinsDiscretizer` and `OrdinalEncoder`, fitted on the training part only (`src/nb_compare.py`). Missing values are one extra category for scikit-learn, while the scratch model skips the attribute for that row. The first check gives both models the same bin codes, so only the counting, smoothing and normalization are compared.

In [ ]:
exact_equivalence(nb_X_tr, nb_y_tr, nb_X_va, NaiveBayes)

In [ ]:
compare_with_sklearn(
    {'Naive Bayes': (NaiveBayes(n_bins=20), SklearnCategoricalNB(n_bins=20))},
    nb_X_tr, nb_y_tr, nb_X_va, nb_y_va,
)

In [ ]:
nb_cv_sklearn = cross_validate(lambda: SklearnCategoricalNB(n_bins=20), prepare_nb, train_df)
pd.concat({'scratch': nb_cv_raw.loc['mean'], 'sklearn': nb_cv_sklearn.loc['mean']}, axis=1).T

### Insights

- With the same bin codes the scratch model and `CategoricalNB` differ by at most 6.5e-15 in probability and predict the same class for all 3,299 complete validation rows. The counting, the Laplace smoothing and the normalization are therefore equivalent.
- On the hold-out validation set the scratch model scores an accuracy of 0.7784, an F1 of 0.6452 and a ROC-AUC of 0.8364 against 0.7790, 0.6460 and 0.8344 for scikit-learn. In 5-fold cross-validation the scores are 0.7812, 0.6462 and 0.8347 against 0.7810, 0.6459 and 0.8336. No metric differs by more than about 0.002.
- The small remaining gap comes from preparation, not from the algorithm. Scikit-learn treats a missing value as a category while the scratch model skips it, and the two discretizers differ for integer attributes with ties: the scratch model keeps the quantile edge at the maximum of `arrival_month`, so month 12 has its own bin, while `KBinsDiscretizer` drops the first and last edge, so months 11 and 12 share a bin.

---

# Saving and Loading Models <a name="9"></a>

---

<!-- artifacts: <model>_v3.pkl -->

This section stores the trained models and restores them for prediction.

In [ ]:
# Write your code here

---

# Kaggle Submission <a name="10"></a>

---

The final pipeline and model are fitted on the original training set, and the test set is only transformed. The submission file contains the `id` and `is_canceled` columns.

In [ ]:
# Write your code here

---

# Error Analysis <a name="11"></a>

---

The analysis supports each step taken in this notebook.